# Phân tích subject shift ở mức input: trước và sau Riemannian alignment (RA)

Notebook này đo **khác biệt tín hiệu EEG giữa các người (subject shift)** trước và sau RA trên cả 3 dataset của bài (BCI IV-2a, BCI IV-2b, Zhou2016). **Không huấn luyện mô hình nào** và **không cần GPU**: runtime CPU là đủ.

**Cách làm (khớp đúng với pipeline của bài):**
- Code load dữ liệu, chia session và RA được lấy trực tiếp từ nhánh `feature/hada-full-mamba-autodial` (`load_bcic4`, `load_zhou2016`, `BaseDataModule._riemannian_align_many`).
- R̄ của mỗi người tính từ **session đầu** (2a: session 1; 2b: session 1–3; Zhou2016: session 1–2), rồi áp nguyên lên **session đánh giá** (2a: session 2; 2b: session 4–5; Zhou2016: session 3).
- **Mọi phép đo chỉ làm trên session đánh giá**, phần RA chưa từng thấy. Trên session đầu, trung bình của mọi người bằng I do cách xây dựng, nên không chứng minh được gì.

**Kết quả lưu vào Google Drive** (`MyDrive/TCFormer-results/ra_shift/`):

| File | Nội dung | Chứng minh được |
|---|---|---|
| `ra_tsne_bcic2a` | t-SNE các trial session 2 của 2a, tô màu theo người, trước và sau RA, kèm silhouette theo người | Trước RA mỗi người là một cụm riêng; sau RA các cụm trộn lẫn, tức RA xóa "dấu vân tay" của từng người |
| `ra_pairwise_bcic2a` | Khoảng cách Riemann 9×9 giữa các người (session 2), trước và sau RA | Khoảng cách giữa các người giảm trên dữ liệu RA chưa thấy |
| `ra_distance_bars` | Khoảng cách từ session đánh giá của mỗi target đến tâm source domain, trước và sau RA, cùng với shift giữa hai session của chính người đó; cho cả 3 dataset | RA giảm mạnh shift giữa các người. Phần còn lại gần bằng **shift giữa các session** (RA không xử lý được phần này), nên cần thêm DA ở mức feature và TTA. So sánh độ cao cột "Before RA" giữa 3 dataset cho biết dataset nào có shift lớn hơn |
| `ra_shift_summary.json`, `ra_shift_results.npz` | Toàn bộ số liệu | Dùng để đưa vào bài |

**Chạy:** bấm *Runtime → Run all*. Lần đầu mất khoảng 15–30 phút, chủ yếu để tải dữ liệu (dữ liệu được lưu lại trên Drive cho các lần sau).

In [ ]:
# 1) Mount Drive, clone the code branch, and install the paper's environment (CPU only).
import os
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import drive

CODE_BRANCH = 'feature/hada-full-mamba-autodial'   # loaders, session splits and RA used in the paper
SCRIPT_BRANCH = 'feature/hada-full-mamba'          # analysis/ra_shift_analysis.py
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/content/tcformer-ra-shift')

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
BNCI_DATA = DRIVE / 'datasets/BNCI_MNE'
ZHOU_DATA = DRIVE / 'datasets/Zhou2016_MNE'
OUTPUT_DIR = DRIVE / 'TCFormer-results/ra_shift'
for path in (BNCI_DATA, ZHOU_DATA, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)


def run(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)


if (REPO_PATH / '.git').is_dir():
    run(['git', 'fetch', 'origin', CODE_BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', '--force', CODE_BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{CODE_BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', CODE_BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

# The analysis script lives on the paper branch; copy it into the code checkout.
run(['git', 'fetch', 'origin', SCRIPT_BRANCH], cwd=REPO_PATH)
script = subprocess.run(['git', 'show', 'FETCH_HEAD:analysis/ra_shift_analysis.py'],
                        cwd=REPO_PATH, check=True, capture_output=True, text=True).stdout
(REPO_PATH / 'ra_shift_analysis.py').write_text(script)
assert '_riemannian_align_many' in (REPO_PATH / 'datamodules/base.py').read_text()

# Same Python 3.10 environment as the training notebooks, with CPU-only PyTorch.
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1',
     '--index-url', 'https://download.pytorch.org/whl/cpu'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
print('Environment ready.')

In [ ]:
# 2) Run the analysis (downloads the datasets on first use; no model is trained).
env = os.environ.copy()
env.update({
    'PYTHONUNBUFFERED': '1',
    'MPLBACKEND': 'Agg',  # the kernel's inline backend is unknown to the venv
    'MNE_DATA': str(BNCI_DATA),
    'MNE_DATASETS_BNCI_PATH': str(BNCI_DATA),
    'MNE_DATASETS_ZHOU2016_PATH': str(ZHOU_DATA),
})
process = subprocess.Popen([str(PYTHON), 'ra_shift_analysis.py', str(OUTPUT_DIR)],
                           cwd=REPO_PATH, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
if process.wait() != 0:
    raise RuntimeError('ra_shift_analysis.py failed; see the output above.')

In [ ]:
# 3) Show the summary and the figures.
import json
import pandas as pd
from IPython.display import Image, Markdown, display

summary = json.loads((OUTPUT_DIR / 'ra_shift_summary.json').read_text())
rows = []
for name, s in summary.items():
    rows.append({
        'Dataset': name,
        'Pairwise distance, before RA': s['pairwise_mean_before'],
        'Pairwise distance, after RA': s['pairwise_mean_after'],
        'Reduction (%)': s['pairwise_reduction_percent'],
        'To source, before RA': sum(s['to_source_before']) / len(s['to_source_before']),
        'To source, after RA': sum(s['to_source_after']) / len(s['to_source_after']),
        'Own session shift': sum(s['session_shift']) / len(s['session_shift']),
        'Silhouette by subject, before': s['silhouette_by_subject_before'],
        'Silhouette by subject, after': s['silhouette_by_subject_after'],
    })
display(pd.DataFrame(rows).set_index('Dataset').round(3))

for stem, title in [
    ('ra_tsne_bcic2a', 'BCI IV-2a evaluation trials, coloured by subject'),
    ('ra_pairwise_bcic2a', 'BCI IV-2a inter-subject distances (evaluation session)'),
    ('ra_distance_bars', 'Distance of each target evaluation session to the source domain'),
]:
    display(Markdown(f'### {title}'))
    display(Image(filename=str(OUTPUT_DIR / f'{stem}.png'), width=900))
print('Files (PNG for viewing, PDF for the paper):', OUTPUT_DIR)

## Cách đọc kết quả

- **Reduction (%)** càng lớn: RA càng giảm khác biệt giữa các người trên session mới.
- **To source, after RA** gần với **Own session shift**: phần shift còn lại chủ yếu là thay đổi giữa hai ngày ghi của chính người đó. RA không thể xử lý phần này (đây là tính chất toán học của RA: nhân cùng một ma trận lên cả hai session thì khoảng cách giữa hai session không đổi), nên cần thêm DA ở mức feature và TTA.
- **Silhouette by subject**: gần 1 nghĩa là các cụm theo người tách biệt rõ, gần 0 nghĩa là các cụm trộn lẫn. Giảm sau RA nghĩa là "dấu vân tay" của từng người yếu đi.
- **So sánh cột "Before RA" giữa 3 dataset**: nếu 2a có shift lớn nhất, đó là bằng chứng giải thích vì sao RA và DA giúp nhiều trên 2a hơn hai dataset còn lại.

Sau khi chạy xong, gửi lại file `ra_shift_summary.json` (và các file PDF nếu cần) để đưa vào bài.